# Gold Model Quality Assurance

Performs final quality assurance across the Gold dimensional model before it is exposed through the Fabric semantic layer.

## Purpose

- Validate Gold dimension and fact row counts
- Confirm dimension-key uniqueness
- Validate fact-table referential integrity
- Confirm fact-grain uniqueness
- Reconcile consumption across Silver and both Gold facts
- Validate tariff-level consumption reconciliation
- Confirm daily completeness outcomes

> These checks provide the final engineering validation checkpoint between Gold-layer construction and semantic modelling.

## 1. Load Gold Model

Load the persisted Gold dimensions and fact tables used by the analytical model.

In [1]:
from pyspark.sql import functions as F

dim_tariff = spark.table("gold.dim_tariff")
dim_household = spark.table("gold.dim_household")
dim_date = spark.table("gold.dim_date")
dim_time = spark.table("gold.dim_time")

fact_daily = spark.table("gold.fact_daily_consumption")
fact_demand = spark.table("gold.fact_demand_pattern")

silver_df = spark.table("silver.meter_readings")

print("Gold QA initialised.")

StatementMeta(, 8b67636d-2efd-40ff-b9e2-ad8fb08d82ef, 3, Finished, Available, Finished, False)

Gold QA initialised.


## 2. Validate Gold Table Counts

Confirm that all expected dimensions and fact tables have been successfully persisted at their designed grains.

In [2]:
print("GOLD TABLE ROW COUNTS")
print("-" * 50)

print(f"dim_tariff:              {dim_tariff.count():,}")
print(f"dim_household:           {dim_household.count():,}")
print(f"dim_date:                {dim_date.count():,}")
print(f"dim_time:                {dim_time.count():,}")
print(f"fact_daily_consumption:  {fact_daily.count():,}")
print(f"fact_demand_pattern:     {fact_demand.count():,}")

StatementMeta(, 8b67636d-2efd-40ff-b9e2-ad8fb08d82ef, 4, Finished, Available, Finished, False)

GOLD TABLE ROW COUNTS
--------------------------------------------------
dim_tariff:              2
dim_household:           5,561
dim_date:                829
dim_time:                48
fact_daily_consumption:  3,510,403
fact_demand_pattern:     79,454


## 3. Validate Dimension Key Uniqueness

Confirm that each Gold dimension contains unique analytical keys before those keys are referenced by the fact tables.

In [3]:
checks = {
    "DimTariff duplicate TariffKey":
        dim_tariff.groupBy("TariffKey").count()
        .filter(F.col("count") > 1).count(),

    "DimHousehold duplicate HouseholdKey":
        dim_household.groupBy("HouseholdKey").count()
        .filter(F.col("count") > 1).count(),

    "DimHousehold duplicate HouseholdID":
        dim_household.groupBy("HouseholdID").count()
        .filter(F.col("count") > 1).count(),

    "DimDate duplicate DateKey":
        dim_date.groupBy("DateKey").count()
        .filter(F.col("count") > 1).count(),

    "DimTime duplicate TimeKey":
        dim_time.groupBy("TimeKey").count()
        .filter(F.col("count") > 1).count()
}

for check_name, result in checks.items():
    print(f"{check_name}: {result}")

StatementMeta(, 8b67636d-2efd-40ff-b9e2-ad8fb08d82ef, 5, Finished, Available, Finished, False)

DimTariff duplicate TariffKey: 0
DimHousehold duplicate HouseholdKey: 0
DimHousehold duplicate HouseholdID: 0
DimDate duplicate DateKey: 0
DimTime duplicate TimeKey: 0


## 4. Validate Daily Fact Referential Integrity

Verify that every dimension key referenced by the Daily Consumption fact resolves to a corresponding Gold dimension record.

In [4]:
missing_household = (
    fact_daily
    .join(
        dim_household.select("HouseholdKey"),
        on="HouseholdKey",
        how="left_anti"
    )
    .count()
)

missing_date = (
    fact_daily
    .join(
        dim_date.select("DateKey"),
        on="DateKey",
        how="left_anti"
    )
    .count()
)

missing_tariff = (
    fact_daily
    .join(
        dim_tariff.select("TariffKey"),
        on="TariffKey",
        how="left_anti"
    )
    .count()
)

print("DAILY FACT REFERENTIAL INTEGRITY")
print("-" * 50)
print(f"Missing Household relationships: {missing_household}")
print(f"Missing Date relationships:      {missing_date}")
print(f"Missing Tariff relationships:    {missing_tariff}")

StatementMeta(, 8b67636d-2efd-40ff-b9e2-ad8fb08d82ef, 6, Finished, Available, Finished, False)

DAILY FACT REFERENTIAL INTEGRITY
--------------------------------------------------
Missing Household relationships: 0
Missing Date relationships:      0
Missing Tariff relationships:    0


## 5. Validate Demand Fact Referential Integrity

Verify that every Date, Time and Tariff key referenced by the Demand Pattern fact resolves to its corresponding Gold dimension.

In [5]:
missing_date_demand = (
    fact_demand
    .join(
        dim_date.select("DateKey"),
        on="DateKey",
        how="left_anti"
    )
    .count()
)

missing_time_demand = (
    fact_demand
    .join(
        dim_time.select("TimeKey"),
        on="TimeKey",
        how="left_anti"
    )
    .count()
)

missing_tariff_demand = (
    fact_demand
    .join(
        dim_tariff.select("TariffKey"),
        on="TariffKey",
        how="left_anti"
    )
    .count()
)

print("DEMAND FACT REFERENTIAL INTEGRITY")
print("-" * 50)
print(f"Missing Date relationships:   {missing_date_demand}")
print(f"Missing Time relationships:   {missing_time_demand}")
print(f"Missing Tariff relationships: {missing_tariff_demand}")

StatementMeta(, 8b67636d-2efd-40ff-b9e2-ad8fb08d82ef, 7, Finished, Available, Finished, False)

DEMAND FACT REFERENTIAL INTEGRITY
--------------------------------------------------
Missing Date relationships:   0
Missing Time relationships:   0
Missing Tariff relationships: 0


## 6. Validate Fact Grain Uniqueness

Confirm that each fact satisfies its intended analytical grain.

- Daily Consumption: `HouseholdKey × DateKey`
- Demand Pattern: `DateKey × TimeKey × TariffKey`

In [6]:
daily_duplicates = (
    fact_daily
    .groupBy(
        "HouseholdKey",
        "DateKey"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

demand_duplicates = (
    fact_demand
    .groupBy(
        "DateKey",
        "TimeKey",
        "TariffKey"
    )
    .count()
    .filter(F.col("count") > 1)
    .count()
)

print("FACT GRAIN VALIDATION")
print("-" * 50)
print(f"Daily fact duplicate keys:  {daily_duplicates}")
print(f"Demand fact duplicate keys: {demand_duplicates}")

StatementMeta(, 8b67636d-2efd-40ff-b9e2-ad8fb08d82ef, 8, Finished, Available, Finished, False)

FACT GRAIN VALIDATION
--------------------------------------------------
Daily fact duplicate keys:  0
Demand fact duplicate keys: 0


## 7. Reconcile Consumption Across Analytical Layers

Compare total electricity consumption across:

1. Validated Silver readings
2. Gold Daily Consumption fact
3. Gold Demand Pattern fact

The two Gold facts use different analytical grains but should preserve the same underlying consumption total.

Minor decimal differences are expected from distributed floating-point aggregation.

In [7]:
silver_total = (
    silver_df
    .agg(F.sum("ConsumptionKWh").alias("Total"))
    .collect()[0]["Total"]
)

daily_total = (
    fact_daily
    .agg(F.sum("DailyConsumptionKWh").alias("Total"))
    .collect()[0]["Total"]
)

demand_total = (
    fact_demand
    .agg(F.sum("TotalConsumptionKWh").alias("Total"))
    .collect()[0]["Total"]
)

print("THREE-WAY CONSUMPTION RECONCILIATION")
print("-" * 50)

print(f"Silver:       {silver_total:,.6f}")
print(f"Gold Daily:   {daily_total:,.6f}")
print(f"Gold Demand:  {demand_total:,.6f}")

print()
print(f"Silver vs Daily difference:  {silver_total - daily_total:,.12f}")
print(f"Silver vs Demand difference: {silver_total - demand_total:,.12f}")

StatementMeta(, 8b67636d-2efd-40ff-b9e2-ad8fb08d82ef, 9, Finished, Available, Finished, False)

THREE-WAY CONSUMPTION RECONCILIATION
--------------------------------------------------
Silver:       35,539,823.306363
Gold Daily:   35,539,823.306385
Gold Demand:  35,539,823.306385

Silver vs Daily difference:  -0.000021956861
Silver vs Demand difference: -0.000021971762


## 8. Reconcile Consumption by Tariff

Validate that consumption remains consistent by household tariff classification after transformation from Silver to the Gold daily fact.

In [8]:
print("DAILY CONSUMPTION BY TARIFF")
print("-" * 50)

(
    fact_daily
    .join(
        dim_tariff,
        on="TariffKey",
        how="inner"
    )
    .groupBy("TariffType")
    .agg(
        F.sum("DailyConsumptionKWh")
        .alias("TotalConsumptionKWh")
    )
    .orderBy("TariffType")
    .show(truncate=False)
)

StatementMeta(, 8b67636d-2efd-40ff-b9e2-ad8fb08d82ef, 10, Finished, Available, Finished, False)

DAILY CONSUMPTION BY TARIFF
--------------------------------------------------
+----------+-------------------+
|TariffType|TotalConsumptionKWh|
+----------+-------------------+
|Std       |2.883344102351394E7|
|ToU       |6706382.282871209  |
+----------+-------------------+



In [9]:
print("SILVER CONSUMPTION BY TARIFF")
print("-" * 50)

(
    silver_df
    .groupBy("TariffType")
    .agg(
        F.sum("ConsumptionKWh")
        .alias("TotalConsumptionKWh")
    )
    .orderBy("TariffType")
    .show(truncate=False)
)

StatementMeta(, 8b67636d-2efd-40ff-b9e2-ad8fb08d82ef, 11, Finished, Available, Finished, False)

SILVER CONSUMPTION BY TARIFF
--------------------------------------------------
+----------+--------------------+
|TariffType|TotalConsumptionKWh |
+----------+--------------------+
|Std       |2.8833441023521014E7|
|ToU       |6706382.282871542   |
+----------+--------------------+



In [10]:
invalid_household_tariff = (
    dim_household
    .filter(
        F.col("TariffKey").isNull()
    )
    .count()
)

print(
    f"Households without tariff key: "
    f"{invalid_household_tariff}"
)

StatementMeta(, 8b67636d-2efd-40ff-b9e2-ad8fb08d82ef, 12, Finished, Available, Finished, False)

Households without tariff key: 0


## 9. Validate Daily Completeness

Confirm that the completeness classification derived during daily aggregation has been preserved in the persisted Gold fact.

In [11]:
print("DAILY COMPLETENESS")
print("-" * 50)

fact_daily.groupBy(
    "IsCompleteDay"
).count().show()

StatementMeta(, 8b67636d-2efd-40ff-b9e2-ad8fb08d82ef, 13, Finished, Available, Finished, False)

DAILY COMPLETENESS
--------------------------------------------------
+-------------+-------+
|IsCompleteDay|  count|
+-------------+-------+
|         true|3469352|
|        false|  41051|
+-------------+-------+



## Gold Model QA Summary

Final validation confirms that the Gold analytical model is internally consistent and reconciles to the validated Silver layer.

- **4** conformed dimensions validated
- **3,510,403** Daily Consumption fact rows
- **79,454** Demand Pattern fact rows
- **0** duplicate dimension keys
- **0** missing dimension references
- **0** duplicate fact-grain keys
- Consumption reconciled across Silver and both Gold facts with negligible floating-point variance
- Tariff-level totals reconciled
- **3,469,352** complete household-days and **41,051** partial household-days retained

The validated Gold model is ready for downstream SQL analysis and Direct Lake semantic modelling.